# Time layers

Any layer whose features carry timestamps can be animated. Timestamps are read
from the layer's **own properties** — a DataFrame's `timestamp` column, or the
interval a geostructures `Track` records — so nothing extra is passed in.

The slider steps through **generated periods**, not observed timestamps: a period
in which nothing happened still gets its tick. An empty map at 03:00 is a finding,
not a gap in the slider.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(2)
steps = 48
frames = []
for vessel, (lat0, lon0) in [("Vessel A", (36.00, -5.80)),
                             ("Vessel B", (35.95, -5.75))]:
    frames.append(pd.DataFrame({
        "lat": lat0 + np.cumsum(rng.normal(0.002, 0.003, steps)),
        "lon": lon0 + np.cumsum(rng.normal(0.009, 0.005, steps)),
        "vessel": vessel,
        "timestamp": pd.date_range("2026-08-01 00:00", periods=steps,
                                   freq="30min", tz="UTC"),
    }))
pings = pd.concat(frames, ignore_index=True)
pings.head(3)

## Animate a layer

The `timestamp` column is found by name (`times`, `datetime_start`/`_end`,
`timestamp`, `datetime`, `time`, `date` are all probed; `time_field=` names
anything else). One slider serves every time layer on the map — animating a second
layer joins it rather than adding another control.

In [ ]:
m = Map()
m.add_circle_markers(pings, name="Pings", color_col="vessel")
m.make_time_layer("Pings", period="PT1H")
m

## What a tick shows: `duration`

- `duration="period"` (default) — each tick shows its own period; absence reads
  as absence.
- `duration=None` — accumulate history instead.
- `duration="PT3H"` — a fixed trailing window.

Re-calling `make_time_layer` on the same target reconfigures it. `fade=True` dims
point features with age — newest at full opacity, zero at the window's trailing
edge.

In [ ]:
m.make_time_layer("Pings", duration="PT3H", fade=True);

## Playback and the shared window

`configure_time` drives the one slider: `period`, `auto_play`, `loop`, `speed`
(ticks per second), `position`, and `window` — a shared trailing window that
overrides every layer's own `duration` while set (the same override dragging the
bar's trail handle creates). `window=None` hands control back.

In [ ]:
m.configure_time(period="PT1H", speed=2, loop=True, position="bottom-center");

## The slider position syncs both ways

`m.time_current` holds the current tick (epoch ms) — reading it is how Shiny
reacts to playback, and setting it jumps the slider:

In [ ]:
import datetime
m.time_current = datetime.datetime(2026, 8, 1, 12, 0,
                                   tzinfo=datetime.timezone.utc).timestamp() * 1000

## Any geometry animates

Lines and polygons carry times the same way — one timestamp (or [start, end]
pair) per feature. Zones that exist for a while and lapse:

In [ ]:
zones = pd.DataFrame({
    "zone_id": ["North"] * 4 + ["South"] * 4,
    "vertex": [0, 1, 2, 3] * 2,
    "lat": [36.10, 36.10, 36.16, 36.16, 35.90, 35.90, 35.96, 35.96],
    "lon": [-5.55, -5.40, -5.40, -5.55, -5.55, -5.40, -5.40, -5.55],
    "times": [["2026-08-01 02:00", "2026-08-01 10:00"]] * 4
           + [["2026-08-01 08:00", "2026-08-01 20:00"]] * 4,
})
m.add_polygon(zones, shape_id_col="zone_id", order_col="vertex",
              name="zone_id", layer_group="Zones", fill_opacity=0.3)
m.make_time_layer(group="Zones");

## Stopping

`clear_time_layer()` returns every layer to always-visible and removes the control
once nothing is animated; a target clears just that layer.

In [ ]:
m.clear_time_layer();

A static export (**08_export**) carries time playback with it — the slider works
in the shared file with no Python behind it.